# Landing Page Performance Analysis

## Project Overview
This notebook analyzes landing page metrics — sessions, bounce rate, conversion rate, and traffic source — to identify which pages perform best and why. We use realistic synthetic data modeled on typical SaaS landing page analytics.

## Learning Objectives
- Compare landing page performance across multiple KPIs simultaneously.
- Decompose conversion funnels by traffic source and page type.
- Identify statistical differences in bounce and conversion rates.
- Translate page-level analytics into actionable optimization recommendations.

## Problem Statement
A digital marketing team runs multiple landing pages across paid, organic, social, and email channels. They need to know which pages convert best, which traffic sources deliver quality visitors, and where to focus optimization efforts.

## Why This Project Matters
Landing pages are the front door of digital acquisition. Small improvements in conversion rate compound into significant revenue gains. Data-driven page optimization is a core skill in growth analytics.

## Dataset Overview
We generate a realistic synthetic dataset of ~5,000 landing page sessions with fields including page name, traffic source, device type, session duration, bounce flag, conversion flag, and page load time.

## Dataset Source and License Notes
Synthetic data generated in-notebook. No external dependencies or licenses required.

## Environment Setup

In [ ]:
import importlib, subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Environment ready.')

## Imports

In [ ]:
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)
np.random.seed(42)

## Configuration / Constants

In [ ]:
PAGES = ['Homepage', 'Pricing', 'Free Trial', 'Demo Request', 'Product Tour',
         'Blog Landing', 'Webinar Signup', 'Case Study']
SOURCES = ['Paid Search', 'Organic Search', 'Social Media', 'Email', 'Direct', 'Referral']
DEVICES = ['Desktop', 'Mobile', 'Tablet']
N_SESSIONS = 5000

## Dataset Generation
We create a realistic landing page dataset. Conversion and bounce probabilities vary by page type, source, and device to simulate real-world patterns.

In [ ]:
rng = np.random.default_rng(42)

# Base conversion rates by page (realistic SaaS ranges)
page_base_cvr = {
    'Homepage': 0.03, 'Pricing': 0.08, 'Free Trial': 0.15, 'Demo Request': 0.12,
    'Product Tour': 0.06, 'Blog Landing': 0.02, 'Webinar Signup': 0.10, 'Case Study': 0.04
}
page_base_bounce = {
    'Homepage': 0.45, 'Pricing': 0.35, 'Free Trial': 0.25, 'Demo Request': 0.30,
    'Product Tour': 0.40, 'Blog Landing': 0.55, 'Webinar Signup': 0.28, 'Case Study': 0.50
}
source_cvr_mult = {'Paid Search': 1.2, 'Organic Search': 1.0, 'Social Media': 0.7,
                   'Email': 1.4, 'Direct': 0.9, 'Referral': 1.1}
device_cvr_mult = {'Desktop': 1.1, 'Mobile': 0.8, 'Tablet': 0.95}

records = []
for _ in range(N_SESSIONS):
    page = rng.choice(PAGES)
    source = rng.choice(SOURCES, p=[0.25, 0.25, 0.15, 0.15, 0.10, 0.10])
    device = rng.choice(DEVICES, p=[0.55, 0.35, 0.10])
    load_time = max(0.5, rng.normal(2.5 if device == 'Desktop' else 3.5, 0.8))

    bounce_prob = page_base_bounce[page] * (1.1 if device == 'Mobile' else 1.0) * (1 + max(0, load_time - 3) * 0.1)
    bounced = rng.random() < min(bounce_prob, 0.85)

    if bounced:
        converted = False
        duration = max(1, rng.exponential(5))
    else:
        cvr = page_base_cvr[page] * source_cvr_mult[source] * device_cvr_mult[device]
        converted = rng.random() < min(cvr, 0.35)
        duration = max(5, rng.exponential(60 if converted else 30))

    date = pd.Timestamp('2025-01-01') + pd.Timedelta(days=int(rng.integers(0, 180)))

    records.append({
        'Date': date, 'Page': page, 'Traffic Source': source, 'Device': device,
        'Page Load Time (s)': round(load_time, 2), 'Session Duration (s)': round(duration, 1),
        'Bounced': int(bounced), 'Converted': int(converted),
    })

df = pd.DataFrame(records)
print(f'Dataset shape: {df.shape}')
df.head()

## Data Validation Checks

In [ ]:
print(f'Rows: {len(df)}, Columns: {len(df.columns)}')
print(f'Nulls:\n{df.isna().sum().to_string()}')
print(f'\nDuplicate rows: {df.duplicated().sum()}')
print(f'\nPages: {sorted(df["Page"].unique())}')
print(f'Sources: {sorted(df["Traffic Source"].unique())}')
print(f'Devices: {sorted(df["Device"].unique())}')
print(f'Date range: {df["Date"].min().date()} to {df["Date"].max().date()}')

## Exploratory Data Analysis

### Overall Metrics
Compute aggregate KPIs across all landing pages.

In [ ]:
overall = pd.Series({
    'Total Sessions': len(df),
    'Overall Bounce Rate %': df['Bounced'].mean() * 100,
    'Overall Conversion Rate %': df['Converted'].mean() * 100,
    'Avg Session Duration (s)': df['Session Duration (s)'].mean(),
    'Avg Page Load Time (s)': df['Page Load Time (s)'].mean(),
    'Conversions': df['Converted'].sum(),
})
print('Overall Metrics:')
print(overall.round(2).to_string())

### Performance by Landing Page

In [ ]:
page_perf = df.groupby('Page').agg(
    Sessions=('Converted', 'count'),
    Bounce_Rate=('Bounced', 'mean'),
    Conversion_Rate=('Converted', 'mean'),
    Conversions=('Converted', 'sum'),
    Avg_Duration=('Session Duration (s)', 'mean'),
    Avg_Load_Time=('Page Load Time (s)', 'mean'),
).sort_values('Conversion_Rate', ascending=False)

page_perf['Bounce_Rate'] *= 100
page_perf['Conversion_Rate'] *= 100

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].barh(page_perf.index, page_perf['Conversion_Rate'], color='seagreen', edgecolor='black')
axes[0].set_xlabel('Conversion Rate %')
axes[0].set_title('Conversion Rate by Page')

axes[1].barh(page_perf.index, page_perf['Bounce_Rate'], color='coral', edgecolor='black')
axes[1].set_xlabel('Bounce Rate %')
axes[1].set_title('Bounce Rate by Page')

axes[2].barh(page_perf.index, page_perf['Sessions'], color='steelblue', edgecolor='black')
axes[2].set_xlabel('Sessions')
axes[2].set_title('Traffic Volume by Page')

plt.tight_layout()
plt.show()

print(page_perf.round(2).to_string())

### Performance by Traffic Source

In [ ]:
source_perf = df.groupby('Traffic Source').agg(
    Sessions=('Converted', 'count'),
    Bounce_Rate=('Bounced', 'mean'),
    Conversion_Rate=('Converted', 'mean'),
    Conversions=('Converted', 'sum'),
    Avg_Duration=('Session Duration (s)', 'mean'),
).sort_values('Conversion_Rate', ascending=False)

source_perf['Bounce_Rate'] *= 100
source_perf['Conversion_Rate'] *= 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(source_perf.index, source_perf['Conversion_Rate'], color='seagreen', edgecolor='black')
axes[0].set_title('Conversion Rate by Source')
axes[0].set_ylabel('CVR %')
axes[0].tick_params(axis='x', rotation=30)

axes[1].bar(source_perf.index, source_perf['Bounce_Rate'], color='coral', edgecolor='black')
axes[1].set_title('Bounce Rate by Source')
axes[1].set_ylabel('Bounce %')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print(source_perf.round(2).to_string())

### Performance by Device

In [ ]:
device_perf = df.groupby('Device').agg(
    Sessions=('Converted', 'count'),
    Bounce_Rate=('Bounced', 'mean'),
    Conversion_Rate=('Converted', 'mean'),
    Avg_Load_Time=('Page Load Time (s)', 'mean'),
).round(3)
device_perf['Bounce_Rate'] *= 100
device_perf['Conversion_Rate'] *= 100

print('Performance by Device:')
print(device_perf.round(2).to_string())

### Page Load Time vs Bounce Rate
Is there a measurable relationship between load time and bouncing?

In [ ]:
load_bins = pd.cut(df['Page Load Time (s)'], bins=[0, 2, 3, 4, 6, 10], labels=['<2s', '2-3s', '3-4s', '4-6s', '6s+'])
load_impact = df.groupby(load_bins, observed=True).agg(
    Sessions=('Bounced', 'count'),
    Bounce_Rate=('Bounced', 'mean'),
    Conversion_Rate=('Converted', 'mean'),
).round(3)
load_impact['Bounce_Rate'] *= 100
load_impact['Conversion_Rate'] *= 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(load_impact.index.astype(str), load_impact['Bounce_Rate'], color='coral', edgecolor='black', alpha=0.7, label='Bounce Rate %')
ax2 = ax.twinx()
ax2.plot(load_impact.index.astype(str), load_impact['Conversion_Rate'], 'go-', linewidth=2, markersize=8, label='CVR %')
ax.set_xlabel('Page Load Time Bucket')
ax.set_ylabel('Bounce Rate %')
ax2.set_ylabel('Conversion Rate %')
ax.set_title('Load Time Impact on Bounce & Conversion')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print(load_impact.round(2).to_string())

### Cross-Dimensional Heatmap: Page × Source Conversion Rate

In [ ]:
cross = df.pivot_table(values='Converted', index='Page', columns='Traffic Source', aggfunc='mean') * 100

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(cross, annot=True, fmt='.1f', cmap='YlGn', ax=ax, linewidths=0.5)
ax.set_title('Conversion Rate (%) — Page × Traffic Source')
plt.tight_layout()
plt.show()

### Weekly Trend

In [ ]:
df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
weekly = df.groupby('Week').agg(
    Sessions=('Converted', 'count'),
    CVR=('Converted', 'mean'),
    Bounce=('Bounced', 'mean'),
).reset_index()
weekly['CVR'] *= 100
weekly['Bounce'] *= 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(weekly['Week'], weekly['Sessions'], color='steelblue', alpha=0.4, label='Sessions')
ax2 = ax.twinx()
ax2.plot(weekly['Week'], weekly['CVR'], 'g-', linewidth=2, label='CVR %')
ax.set_xlabel('Week')
ax.set_ylabel('Sessions')
ax2.set_ylabel('CVR %')
ax.set_title('Weekly Sessions and Conversion Rate')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

### Statistical Test: Top Page vs Average
Is the highest-converting page significantly better than the dataset average?

In [ ]:
best_page = page_perf['Conversion_Rate'].idxmax()
best_data = df[df['Page'] == best_page]['Converted']
rest_data = df[df['Page'] != best_page]['Converted']

stat, pval = stats.mannwhitneyu(best_data, rest_data, alternative='greater')
print(f'Best page: {best_page}')
print(f'  CVR: {best_data.mean()*100:.2f}% vs rest: {rest_data.mean()*100:.2f}%')
print(f'  Mann-Whitney U p-value: {pval:.4f}')
print(f'  Statistically significant (p < 0.05): {pval < 0.05}')

## Executive Summary Insights

In [ ]:
best_page_name = page_perf['Conversion_Rate'].idxmax()
worst_page_name = page_perf['Conversion_Rate'].idxmin()
best_source = source_perf['Conversion_Rate'].idxmax()

insights = [
    f'Total sessions analyzed: {len(df):,}',
    f'Overall conversion rate: {df["Converted"].mean()*100:.2f}%',
    f'Overall bounce rate: {df["Bounced"].mean()*100:.2f}%',
    f'Best converting page: {best_page_name} ({page_perf.loc[best_page_name, "Conversion_Rate"]:.1f}%)',
    f'Worst converting page: {worst_page_name} ({page_perf.loc[worst_page_name, "Conversion_Rate"]:.1f}%)',
    f'Best traffic source: {best_source} ({source_perf.loc[best_source, "Conversion_Rate"]:.1f}% CVR)',
    f'Mobile bounce rate is higher than desktop — load time is a key factor',
    f'Pages loading > 4s see significantly higher bounce rates',
]
for item in insights:
    print(f'  * {item}')

## Limitations
- Synthetic data — real landing page data would show more complex patterns and seasonality.
- No revenue or cost data attached to conversions, so ROI comparisons are not possible.
- No A/B test structure — differences are observational, not causal.
- Session-level data only; no user-level journey tracking.

## Recommendations
- Prioritize page load time optimization, especially for mobile visitors.
- Investigate why Blog Landing and Case Study pages have high bounce — likely intent mismatch.
- Increase budget for traffic sources with highest CVR (Email, Paid Search).
- Run A/B tests on the Free Trial and Demo Request pages to push CVR higher.

## Common Mistakes
- Optimizing for conversion rate without checking traffic volume — a 20% CVR on 10 sessions is noise.
- Ignoring device-specific performance — mobile often needs separate optimization.
- Treating all traffic sources equally — source intent quality varies enormously.
- Not accounting for page load time as a confound when comparing page performance.

## Mini Challenge
1. Calculate the "quality score" for each traffic source as CVR × avg session duration, and rank them.
2. Build a cohort analysis: do conversion rates improve over the 6-month period?
3. Find which page × device combination has the worst bounce rate and propose a fix.

## Final Summary / Key Takeaways
Landing page performance analysis requires looking at multiple metrics together: traffic volume, bounce rate, conversion rate, load time, and source quality. The best pages combine fast load times, matched visitor intent, and clear calls to action. Always segment by device and traffic source before drawing conclusions.